In [ ]:
from IPython.display import Audio
import numpy as np
from pathlib import Path
import soundfile as sf
from datasets import Dataset
import os
import gc
import json
import math
from itertools import groupby
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig
from scipy.signal import butter, sosfilt, iirnotch, lfilter, stft, istft
from pystoi import stoi

c:\Users\Admin\Desktop\sound_ml_lab1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset
TIMIT is a classic speech dataset for ASR and phoneme-recognition research. It contains recordings of English read speech from 630 speakers across 8 major American English dialect regions, with 10 sentences per speaker. It includes 16-bit, 16 kHz single-channel audio plus time-aligned orthographic, word-level, and phonetic transcriptions.

In [ ]:

sample_dir = Path('data/data/TEST/DR1/FAKS0')
utterance = 'SA1'

txt_path = sample_dir / f'{utterance}.TXT'
wrd_path = sample_dir / f'{utterance}.WRD'
phn_path = sample_dir / f'{utterance}.PHN'
wav_path = sample_dir / f'{utterance}.WAV.wav'

print(f"Speaker: FAKS0 | Dialect Region: DR1 | Set: TEST | Utterance: {utterance}")
print(f"Files: {', '.join(os.listdir(sample_dir)[:5])}... ({len(os.listdir(sample_dir))} total)\n")

print("=== Sentence (.TXT) ===")
print(txt_path.read_text().strip())

print("\n=== Word alignment (.WRD) ===")
print(wrd_path.read_text().strip())

print("\n=== Phoneme alignment (.PHN) ===")
print(phn_path.read_text().strip())

print("\n=== Audio (.WAV) ===")
audio_data, sr = sf.read(wav_path)
print(f"Sample rate: {sr} Hz | Duration: {len(audio_data)/sr:.2f}s | Samples: {len(audio_data)}")
Audio(audio_data, rate=sr)


Speaker: FAKS0 | Dialect Region: DR1 | Set: TEST | Utterance: SA1
Files: SA1.PHN, SA1.TXT, SA1.WAV, SA1.WAV.wav, SA1.WRD... (50 total)

=== Sentence (.TXT) ===
0 63488 She had your dark suit in greasy wash water all year.

=== Word alignment (.WRD) ===
9640 12783 she
12783 17103 had
17103 18760 your
18760 24104 dark
24104 29179 suit
29179 31880 in
31880 38568 greasy
38568 45119 wash
45624 51033 water
52378 55461 all
55461 60600 year

=== Phoneme alignment (.PHN) ===
0 9640 h#
9640 11240 sh
11240 12783 iy
12783 14078 hv
14078 16157 ae
16157 16880 dcl
16880 17103 d
17103 17587 y
17587 18760 er
18760 19720 dcl
19720 19962 d
19962 21514 aa
21514 22680 r
22680 23800 kcl
23800 24104 k
24104 26280 s
26280 28591 uw
28591 29179 dx
29179 30337 ih
30337 31880 ng
31880 32500 gcl
32500 33170 g
33170 33829 r
33829 35150 iy
35150 37370 s
37370 38568 iy
38568 40546 w
40546 42357 aa
42357 45119 sh
45119 45624 epi
45624 46855 w
46855 48680 aa
48680 49240 dx
49240 51033 er
51033 52378 q
52378 54500 ao
54

In [ ]:

class TimitDataset:
    """Lazy-loading dataset: stores paths, reads audio on access."""
    def __init__(self, data_root):
        self.records = []
        for split_dir in ['TEST', 'TRAIN']:
            split_path = Path(data_root) / split_dir
            if not split_path.exists():
                continue
            for wav_file in sorted(split_path.rglob('*.WAV.wav')):
                self.records.append({
                    'speaker': wav_file.parent.name,
                    'utterance': wav_file.stem.replace('.WAV', ''),
                    'split': split_dir,
                    'path': str(wav_file),
                })

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        audio, sr = sf.read(rec['path'])
        return {**rec, 'audio': audio, 'sampling_rate': sr}

data = TimitDataset('data/data')
print(f"Dataset: {len(data)} samples")
sample = data[0]
print(f"Example: sr={sample['sampling_rate']}, audio shape={sample['audio'].shape}")
print(f"Speaker: {sample['speaker']}, Utterance: {sample['utterance']}, Split: {sample['split']}")


Dataset: 6300 samples
Example: sr=16000, audio shape=(63488,)
Speaker: FAKS0, Utterance: SA1, Split: TEST


In [ ]:
sample = data[0]
Audio(sample['audio'], rate=sample['sampling_rate'])

-----------------------------------------------

## Simple Filters

### High-Pass Filter
*removes very low-frequency noise*

In [ ]:
def highpass_filter(audio, sr, cutoff=80):
    sos = butter(5, cutoff, 'hp', fs=sr, output='sos')
    return sosfilt(sos, audio)

In [ ]:
audio_before = data[0]['audio']
sr = data[0]['sampling_rate']

audio_after = highpass_filter(audio_before, sr, cutoff=80)

print(f"Before: min={audio_before.min():.4f}, max={audio_before.max():.4f}, mean={audio_before.mean():.6f}")
print(f"After:  min={audio_after.min():.4f}, max={audio_after.max():.4f}, mean={audio_after.mean():.6f}")

print("\nBefore high-pass filter:")
Audio(audio_before, rate=sr)


Before: min=-0.2126, max=0.2353, mean=0.000003
After:  min=-0.2220, max=0.2253, mean=0.000000

Before high-pass filter:


In [ ]:
print("After high-pass filter (80 Hz cutoff):")
Audio(audio_after, rate=sr)

After high-pass filter (80 Hz cutoff):


### Low-Pass Filter
*removes very high-frequency noise*

In [10]:
def lowpass_filter(audio, sr, cutoff=7000):
    sos = butter(5, cutoff, 'lp', fs=sr, output='sos')
    return sosfilt(sos, audio)

In [ ]:
audio_before_lp = data[0]['audio']
sr = data[0]['sampling_rate']

audio_after_lp = lowpass_filter(audio_before_lp, sr, cutoff=7000)

print(f"Before: min={audio_before_lp.min():.4f}, max={audio_before_lp.max():.4f}, mean={audio_before_lp.mean():.6f}")
print(f"After:  min={audio_after_lp.min():.4f}, max={audio_after_lp.max():.4f}, mean={audio_after_lp.mean():.6f}")

print("\nBefore low-pass filter:")
Audio(audio_before_lp, rate=sr)


Before: min=-0.2126, max=0.2353, mean=0.000003
After:  min=-0.2126, max=0.2325, mean=0.000003

Before low-pass filter:


In [12]:
print("After low-pass filter (7000 Hz cutoff):")
Audio(audio_after_lp, rate=sr)

After low-pass filter (7000 Hz cutoff):


### Band-Pass Filter
*combination of two previous, keeps only the main speech frequency range*

In [13]:
def bandpass_filter(audio, sr, low_cutoff=80, high_cutoff=7000):
    sos = butter(5, [low_cutoff, high_cutoff], 'bp', fs=sr, output='sos')
    return sosfilt(sos, audio)

In [ ]:
audio_before_bp = data[0]['audio']
sr = data[0]['sampling_rate']

audio_after_bp = bandpass_filter(audio_before_bp, sr, low_cutoff=80, high_cutoff=7000)

print(f"Before: min={audio_before_bp.min():.4f}, max={audio_before_bp.max():.4f}, mean={audio_before_bp.mean():.6f}")
print(f"After:  min={audio_after_bp.min():.4f}, max={audio_after_bp.max():.4f}, mean={audio_after_bp.mean():.6f}")

print("\nBefore band-pass filter:")
Audio(audio_before_bp, rate=sr)


Before: min=-0.2126, max=0.2353, mean=0.000003
After:  min=-0.2211, max=0.2272, mean=0.000000

Before band-pass filter:


In [15]:
print("After band-pass filter (80-7000 Hz):")
Audio(audio_after_bp, rate=sr)

After band-pass filter (80-7000 Hz):


### Notch Filter
*removes one specific frequency, like 50 Hz hum*

In [ ]:
def notch_filter(audio, sr, freq=50, q=30):
    b, a = iirnotch(freq, q, sr)
    return lfilter(b, a, audio)

In [ ]:
audio_before_notch = data[0]['audio']
sr = data[0]['sampling_rate']

audio_after_notch = notch_filter(audio_before_notch, sr, freq=50, q=30)

print(f"Before: min={audio_before_notch.min():.4f}, max={audio_before_notch.max():.4f}, mean={audio_before_notch.mean():.6f}")
print(f"After:  min={audio_after_notch.min():.4f}, max={audio_after_notch.max():.4f}, mean={audio_after_notch.mean():.6f}")

print("\nBefore notch filter:")
Audio(audio_before_notch, rate=sr)


Before: min=-0.2126, max=0.2353, mean=0.000003
After:  min=-0.2126, max=0.2353, mean=0.000003

Before notch filter:


In [18]:
print("After notch filter (50 Hz removed):")
Audio(audio_after_notch, rate=sr)

After notch filter (50 Hz removed):


In [ ]:
def combined_filter(audio, sr, hp_cutoff=633, lp_cutoff=7000, notch_freq=120):
    """Apply all filters together: high-pass, low-pass, and notch"""
    audio = highpass_filter(audio, sr, cutoff=hp_cutoff)
    audio = notch_filter(audio, sr, freq=notch_freq, q=30)
    audio = lowpass_filter(audio, sr, cutoff=lp_cutoff)
    return audio

audio_original = data[0]['audio']
sr = data[0]['sampling_rate']
audio_combined = combined_filter(audio_original, sr, hp_cutoff=633, lp_cutoff=7000, notch_freq=120)

print("Combined Filter (High-pass 633Hz + Notch 120Hz + Low-pass 7000Hz)")
print(f"Before: min={audio_original.min():.4f}, max={audio_original.max():.4f}")
print(f"After:  min={audio_combined.min():.4f}, max={audio_combined.max():.4f}")

print("\nBefore combined filtering:")
Audio(audio_original, rate=sr)

Combined Filter (High-pass 633Hz + Notch 120Hz + Low-pass 7000Hz)
Before: min=-0.2126, max=0.2353
After:  min=-0.1689, max=0.2096

Before combined filtering:


In [20]:
print("After combined filtering:")
Audio(audio_combined, rate=sr)

After combined filtering:


------------------------------------------------------------


### Pre-Emphasis Filter
*boosts high frequencies to improve speech clarity*

In [ ]:
def pre_emphasis_filter(audio, alpha=0.97):
    y = audio.copy()
    y[1:] = audio[1:] - alpha * audio[:-1]
    return y

In [22]:
audio_before_pe = data[0]['audio']
sr = data[0]['sampling_rate']

audio_after_pe = pre_emphasis_filter(audio_before_pe, alpha=0.6)

print(f"Before: min={audio_before_pe.min():.4f}, max={audio_before_pe.max():.4f}, mean={audio_before_pe.mean():.6f}")
print(f"After:  min={audio_after_pe.min():.4f}, max={audio_after_pe.max():.4f}, mean={audio_after_pe.mean():.6f}")

print("\nBefore pre-emphasis filter:")
Audio(audio_before_pe, rate=sr)

Before: min=-0.2126, max=0.2353, mean=0.000003
After:  min=-0.1488, max=0.1469, mean=0.000001

Before pre-emphasis filter:


In [23]:
print("After pre-emphasis filter (alpha=0.97):")
Audio(audio_after_pe, rate=sr)

After pre-emphasis filter (alpha=0.97):


---------------------------------------------------------

### Spectral Subtraction
*removes noise by subtracting estimated noise spectrum from signal spectrum*

In [ ]:
def spectral_subtraction(audio, sr):
    f, t, X = stft(audio, fs=sr)
    noise_profile = np.mean(np.abs(X[:, :5]), axis=1, keepdims=True)
    magnitude = np.abs(X)
    phase = np.angle(X)
    magnitude_sub = np.maximum(magnitude - noise_profile, 0)
    X_sub = magnitude_sub * np.exp(1j * phase)
    _, audio_sub = istft(X_sub, fs=sr)
    return audio_sub

In [25]:
audio_before_ss = data[0]['audio']
sr = data[0]['sampling_rate']

audio_after_ss = spectral_subtraction(audio_before_ss, sr)

print(f"Before: min={audio_before_ss.min():.4f}, max={audio_before_ss.max():.4f}, mean={audio_before_ss.mean():.6f}")
print(f"After:  min={audio_after_ss.min():.4f}, max={audio_after_ss.max():.4f}, mean={audio_after_ss.mean():.6f}")

print("\nBefore spectral subtraction:")
Audio(audio_before_ss, rate=sr)

Before: min=-0.2126, max=0.2353, mean=0.000003
After:  min=-0.2127, max=0.2353, mean=-0.000001

Before spectral subtraction:


In [26]:
print("After spectral subtraction:")
Audio(audio_after_ss, rate=sr)

After spectral subtraction:


---------------------------------------------------

### Wiener Filter
*adaptive filtering using estimated noise and signal power*

In [27]:
def wiener_filter_audio(audio, sr):
    f, t, X = stft(audio, fs=sr)
    noise_power = np.mean(np.abs(X[:, :5])**2, axis=1, keepdims=True)
    signal_power = np.mean(np.abs(X)**2, axis=1, keepdims=True)
    gain = np.maximum((signal_power - noise_power) / (signal_power + 1e-10), 0)
    magnitude = np.abs(X)
    phase = np.angle(X)
    magnitude_wiener = magnitude * gain
    X_wiener = magnitude_wiener * np.exp(1j * phase)
    _, audio_wiener = istft(X_wiener, fs=sr)
    return audio_wiener

In [28]:
audio_before_wf = data[0]['audio']
sr = data[0]['sampling_rate']

audio_after_wf = wiener_filter_audio(audio_before_wf, sr)

print(f"Before: min={audio_before_wf.min():.4f}, max={audio_before_wf.max():.4f}, mean={audio_before_wf.mean():.6f}")
print(f"After:  min={audio_after_wf.min():.4f}, max={audio_after_wf.max():.4f}, mean={audio_after_wf.mean():.6f}")

print("\nBefore Wiener filter:")
Audio(audio_before_wf, rate=sr)

Before: min=-0.2126, max=0.2353, mean=0.000003
After:  min=-0.2127, max=0.2353, mean=-0.000000

Before Wiener filter:


In [29]:
print("After Wiener filter:")
Audio(audio_after_wf, rate=sr)

After Wiener filter:


-----------------------------------

In [ ]:
filter_dirs = {
    'combined': Path('filters/combined'),
    'pre_emphasis': Path('filters/pre_emphasis'),
    'spectral_subtraction': Path('filters/spectral_subtraction'),
    'wiener_filter': Path('filters/wiener_filter'),
    'original': Path('filters/original'),
}

for filter_name, filter_path in filter_dirs.items():
    filter_path.mkdir(parents=True, exist_ok=True)

print(f'✓ Created filter directories')
print(f'  Total samples to process: {len(data)}')


✓ Created filter directories
  Total samples to process: 6300


In [ ]:
def save_filtered_audio(sample_idx, sample, filter_name, filtered_audio):
    """Save filtered audio with proper directory structure"""
    speaker = sample['speaker']
    utterance = sample['utterance']
    split = sample['split']
    
    dir_path = Path('filters') / filter_name / split / speaker
    dir_path.mkdir(parents=True, exist_ok=True)
    
    file_path = dir_path / f'{speaker}_{utterance}.wav'
    sf.write(file_path, filtered_audio, sample['sampling_rate'])
    return file_path

filter_functions = {
    'original': lambda audio, sr: audio,
    'combined': lambda audio, sr: combined_filter(audio, sr, hp_cutoff=633, lp_cutoff=7000, notch_freq=120),
    'pre_emphasis': lambda audio, sr: pre_emphasis_filter(audio, alpha=0.97),
    'spectral_subtraction': lambda audio, sr: spectral_subtraction(audio, sr),
    'wiener_filter': lambda audio, sr: wiener_filter_audio(audio, sr),
}

print("Processing and saving all filtered variants...")
print("="*80)

filter_counts = {name: 0 for name in filter_functions.keys()}

for sample_idx in range(len(data)):
    if (sample_idx + 1) % max(1, len(data) // 10) == 0:
        print(f"  Progress: {sample_idx + 1}/{len(data)} samples")
    
    sample = data[sample_idx]
    audio = sample['audio']
    sr = sample['sampling_rate']
    
    for filter_name, filter_func in filter_functions.items():
        try:
            filtered_audio = filter_func(audio, sr)
            save_filtered_audio(sample_idx, sample, filter_name, filtered_audio)
            filter_counts[filter_name] += 1
        except Exception as e:
            print(f"  ⚠ Error with {filter_name} on sample {sample_idx}: {str(e)[:50]}")

gc.collect()

print("="*80)
print("✓ Preprocessing complete!\n")

print("Filter results saved:")
for filter_name, count in filter_counts.items():
    print(f"  {filter_name:25s}: {count:5d} files")


Processing and saving all filtered variants...


KeyboardInterrupt: 

In [ ]:
class FilteredDataset:
    """Load pre-filtered audio from saved directories"""
    def __init__(self, filter_name):
        self.filter_name = filter_name
        self.records = []
        filter_path = Path('filters') / filter_name
        
        if not filter_path.exists():
            print(f"⚠ {filter_path} not found - preprocess first")
            return
        
        for wav_file in sorted(filter_path.rglob('*.wav')):
            parts = wav_file.parts
            speaker = wav_file.parent.name
            utterance = wav_file.stem.split('_')[1] if '_' in wav_file.stem else wav_file.stem
            split = wav_file.parent.parent.name
            
            self.records.append({
                'speaker': speaker,
                'utterance': utterance,
                'split': split,
                'filter': filter_name,
                'path': str(wav_file),
            })
    
    def __len__(self):
        return len(self.records)
    
    def __getitem__(self, idx):
        rec = self.records[idx]
        audio, sr = sf.read(rec['path'])
        return {**rec, 'audio': audio, 'sampling_rate': sr}

filtered_datasets = {}
print("\nLoading filtered datasets:")
for filter_name in filter_functions.keys():
    dataset = FilteredDataset(filter_name)
    filtered_datasets[filter_name] = dataset
    print(f"  {filter_name:25s}: {len(dataset):5d} samples")

print(f"\n✓ Ready for inference with filtered audio!")



Loading filtered datasets:
  original                 :  6300 samples
  combined                 :  6300 samples
  pre_emphasis             :  6300 samples
  spectral_subtraction     :  6300 samples
  wiener_filter            :  6300 samples

✓ Ready for inference with filtered audio!



---

## Inference with Trained Model

Load the trained phoneme classifier and run inference on audio samples


In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cpu


In [ ]:
PHONEME_MAP = {
    'aa': 'aa', 'ae': 'ae', 'ah': 'ah', 'ao': 'aa', 'aw': 'aw', 'ax': 'ah', 'ax-h': 'ah', 'axr': 'er', 'ay': 'ay', 'b': 'b',
    'bcl': 'sil', 'ch': 'ch', 'd': 'd', 'dcl': 'sil', 'dh': 'dh', 'dx': 'dx', 'eh': 'eh', 'el': 'l', 'em': 'm', 'en': 'n',
    'eng': 'ng', 'epi': 'sil', 'er': 'er', 'ey': 'ey', 'f': 'f', 'g': 'g', 'gcl': 'sil', 'h#': 'sil', 'hh': 'hh', 'hv': 'hh',
    'ih': 'ih', 'ix': 'ih', 'iy': 'iy', 'jh': 'jh', 'k': 'k', 'kcl': 'sil', 'l': 'l', 'm': 'm', 'n': 'n', 'ng': 'ng',
    'nx': 'n', 'ow': 'ow', 'oy': 'oy', 'p': 'p', 'pau': 'sil', 'pcl': 'sil', 'q': 'sil', 'r': 'r', 's': 's', 'sh': 'sh',
    't': 't', 'tcl': 'sil', 'th': 'th', 'uh': 'uh', 'uw': 'uw', 'ux': 'uw', 'v': 'v', 'w': 'w', 'y': 'y', 'z': 'z', 'zh': 'sh'
}

unique_phonemes_39 = sorted(list(set(PHONEME_MAP.values())))

BLANK_IDX = 0
phoneme_to_id = {'<blank>': BLANK_IDX}
id_to_phoneme = {BLANK_IDX: '<blank>'}

for i, phn in enumerate(unique_phonemes_39, start=1):
    phoneme_to_id[phn] = i
    id_to_phoneme[i] = phn

NUM_CLASSES = 40

print(f'Number of phoneme classes: {NUM_CLASSES}')

print(f'Number of phoneme classes: {NUM_CLASSES}')
print(f'Phonemes: {unique_phonemes_39}')
print(f'ID mappings created with blank at index {BLANK_IDX}')

Number of phoneme classes: 40
Phonemes: ['aa', 'ae', 'ah', 'aw', 'ay', 'b', 'ch', 'd', 'dh', 'dx', 'eh', 'er', 'ey', 'f', 'g', 'hh', 'ih', 'iy', 'jh', 'k', 'l', 'm', 'n', 'ng', 'ow', 'oy', 'p', 'r', 's', 'sh', 'sil', 't', 'th', 'uh', 'uw', 'v', 'w', 'y', 'z']
ID mappings created with blank at index 0


In [ ]:
class SSLPhonemeClassifierCTC(nn.Module):
    """Classifier for phoneme recognition using SSL models + CTC Loss"""
    def __init__(self, model_name, num_classes=40, layer_index=None, freeze_encoder=True):
        super().__init__()
        self.layer_index = layer_index
        self.model_name = model_name

        print(f'Loading {model_name}...')
        self.ssl_model = AutoModel.from_pretrained(model_name, output_hidden_states=True)

        if freeze_encoder:
            for param in self.ssl_model.parameters():
                param.requires_grad = False

        hidden_size = self.ssl_model.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, audio, attention_mask=None):
        outputs = self.ssl_model(audio, attention_mask=attention_mask)

        if self.layer_index is None or self.layer_index == -1:
            hidden = outputs.last_hidden_state
        else:
            all_hidden = outputs.hidden_states
            hidden = all_hidden[self.layer_index + 1]

        logits = self.classifier(hidden)
        return logits


def compute_encoder_output_length(audio_length: int) -> int:
    """Compute number of frames after CNN encoder"""
    strides = [5, 2, 2, 2, 2, 2, 2]
    kernels = [10, 3, 3, 3, 3, 2, 2]
    length = audio_length
    for kernel, stride in zip(kernels, strides):
        length = (length - kernel) // stride + 1
    return max(1, length)


print('Model architecture ready')


Model architecture ready


In [ ]:
def greedy_ctc_decode(log_probs_batch):
    """Greedy CTC decoding: argmax + collapse duplicates + remove blanks"""
    preds = torch.argmax(log_probs_batch, dim=-1)
    decoded_batch = []
    for seq in preds:
        seq = seq.cpu().tolist()
        collapsed = [k for k, _ in groupby(seq)]
        decoded = [x for x in collapsed if x != BLANK_IDX]
        decoded_batch.append(decoded)
    return decoded_batch


@torch.no_grad()
def predict_phonemes(model, audio_array, sr=16000, return_logits=False):
    """
    Predict phonemes for audio.
    
    Returns:
        predicted_phonemes: list of phoneme strings
        logits: [1, T, num_classes] if return_logits=True
    """
    model.eval()
    
    if isinstance(audio_array, np.ndarray):
        audio_tensor = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0)
    else:
        audio_tensor = audio_array.unsqueeze(0) if audio_array.dim() == 1 else audio_array
    
    audio_tensor = audio_tensor.to(device)
    logits = model(audio_tensor)
    
    decoded = greedy_ctc_decode(logits)
    predicted_ids = decoded[0]
    predicted_phonemes = [id_to_phoneme[idx] for idx in predicted_ids]
    
    if return_logits:
        return predicted_phonemes, logits
    return predicted_phonemes


# Load checkpoint from parent directory or current directory
checkpoint = None
checkpoint_path = None

paths_to_try = [
    Path('../checkpoint_results.json'),
    Path('checkpoint_results.json'),
    Path('Task2/checkpoint_results.json')
]

for path in paths_to_try:
    if path.exists():
        checkpoint_path = path
        with open(checkpoint_path) as f:
            checkpoint = json.load(f)
        print(f'✓ Loaded checkpoint from: {checkpoint_path}')
        break

if checkpoint is None:
    print('⚠ Checkpoint not found, will initialize model with default configuration')
    checkpoint = {}


✓ Loaded checkpoint from: checkpoint_results.json


In [ ]:
best_config = None
best_per = float('inf')

if checkpoint:
    print("="*80)
    print("CHECKPOINT RESULTS FOUND")
    print("="*80)
    
    # Find best result (excluding baseline, only CTC results)
    for key, result in checkpoint.items():
        if isinstance(result, dict) and 'per' in result and 'model' in result:
            per = result['per']
            model_name = result['model']
            layer = result.get('layer', 'last')
            
            print(f"\n{key}:")
            print(f"  Model: {model_name}")
            print(f"  Layer: {layer}")
            print(f"  PER: {per:.2f}%")
            
            if per < best_per:
                best_per = per
                best_config = result
    
    if best_config:
        print(f"\n{'='*80}")
        print(f"BEST MODEL SELECTED")
        print(f"{'='*80}")
        print(f"Model: {best_config['model']}")
        print(f"Layer: {best_config.get('layer', 'last')}")
        print(f"PER: {best_per:.2f}%")
        print(f"{'='*80}\n")
        
        model_name = best_config['model']
        layer_idx = best_config.get('layer', None)
        
        if isinstance(layer_idx, str):
            if layer_idx == 'last':
                layer_idx = None
            else:
                try:
                    layer_idx = int(layer_idx.split('_')[-1]) - 1
                except:
                    layer_idx = None
else:
    print("⚠ No checkpoint found, using default configuration")
    model_name = 'facebook/hubert-base-ls960'
    layer_idx = None

print(f'Loading model: {model_name}')
print(f'  Layer: {layer_idx if layer_idx is not None else "last_hidden_state"}')
print(f'  Loss: CTC')
print(f'  Freeze encoder: True\n')

best_model = SSLPhonemeClassifierCTC(
    model_name=model_name,
    num_classes=NUM_CLASSES,
    layer_index=layer_idx,
    freeze_encoder=True
)
best_model.to(device)
print('✓ Model loaded successfully!')



CHECKPOINT RESULTS FOUND

result_w2v2_last:
  Model: facebook/wav2vec2-base
  Layer: None
  PER: 60.59%

result_hubert_last:
  Model: facebook/hubert-base-ls960
  Layer: None
  PER: 21.86%

result_wavlm_last:
  Model: microsoft/wavlm-base
  Layer: None
  PER: 19.05%

BEST MODEL SELECTED
Model: microsoft/wavlm-base
Layer: None
PER: 19.05%

Loading model: microsoft/wavlm-base
  Layer: last_hidden_state
  Loss: CTC
  Freeze encoder: True

Loading microsoft/wavlm-base...


Loading weights: 100%|██████████| 248/248 [00:00<00:00, 14993.48it/s]

✓ Model loaded successfully!


------------------------------------

In [ ]:
def load_checkpoint_weights(model, checkpoint_path=None):
    """
    Load checkpoint weights into the model.
    If checkpoint_path is JSON, looks for corresponding .pt/.pth in same directory.
    Otherwise searches for .pt or .pth files directly.
    """
    model.eval()
    
    weights_path = None
    
    if checkpoint_path is None:
        if Path('checkpoint_results.json').exists():
            checkpoint_dir = Path('checkpoint_results.json').parent
        else:
            checkpoint_dir = Path('.')
    else:
        checkpoint_path = Path(checkpoint_path)
        checkpoint_dir = checkpoint_path.parent
    
    # If checkpoint is JSON, look for weights in same directory
    if checkpoint_path and str(checkpoint_path).endswith('.json'):
        weights_files = list(checkpoint_dir.glob('*.pt')) + list(checkpoint_dir.glob('*.pth'))
        if weights_files:
            weights_path = weights_files[0]
            print(f'Found weights file: {weights_path}')
    else:
        # Search for checkpoint weight files
        weights_files = list(checkpoint_dir.glob('*.pt')) + list(checkpoint_dir.glob('*.pth'))
        weights_files += list(checkpoint_dir.parent.glob('*.pt')) + list(checkpoint_dir.parent.glob('*.pth'))
        if weights_files:
            weights_path = weights_files[0]
            print(f'Found checkpoint: {weights_path}')
    
    if weights_path is None:
        print('⚠ No model weights file found (.pt/.pth), using pretrained model from HuggingFace')
        return model
    
    if not weights_path.exists():
        print(f'⚠ Weights file not found at {weights_path}')
        return model
    
    # Load checkpoint
    try:
        checkpoint = torch.load(weights_path, map_location=device)
        
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                state = checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint:
                state = checkpoint['state_dict']
            else:
                state = checkpoint
        else:
            state = checkpoint
        
        model.load_state_dict(state, strict=False)
        print(f'✓ Model weights loaded successfully from {weights_path}')
    except Exception as e:
        print(f'⚠ Could not load weights: {e}')
    
    return model


def inference_on_filtered_dataset(filter_name, model, max_samples=None):
    """
    Run inference on all samples from a filtered dataset variant.
    
    Returns:
        results: dict with speaker, utterance, split, and predicted_phonemes
    """
    dataset = filtered_datasets.get(filter_name)
    if dataset is None or len(dataset) == 0:
        print(f'⚠ Dataset {filter_name} not available')
        return []
    
    results = []
    num_samples = min(len(dataset), max_samples) if max_samples else len(dataset)
    
    print(f'Running inference on {filter_name} ({num_samples} samples)...')
    
    for idx in range(num_samples):
        if (idx + 1) % max(1, num_samples // 10) == 0:
            print(f'  Progress: {idx + 1}/{num_samples}')
        
        sample = dataset[idx]
        audio = sample['audio']
        
        predicted_phonemes = predict_phonemes(model, audio)
        
        results.append({
            'speaker': sample['speaker'],
            'utterance': sample['utterance'],
            'split': sample['split'],
            'filter': filter_name,
            'predicted_phonemes': predicted_phonemes,
            'num_phonemes': len(predicted_phonemes)
        })
    
    print(f'✓ Inference complete on {filter_name}\n')
    return results

print('✓ Checkpoint loading and inference functions defined')

✓ Checkpoint loading and inference functions defined


In [ ]:
print('Attempting to load checkpoint weights...')
best_model = load_checkpoint_weights(best_model, Path('wavlm_classifier.pt'))
print()

Attempting to load checkpoint weights...
Found checkpoint: wavlm_classifier.pt
✓ Model weights loaded successfully from wavlm_classifier.pt



In [ ]:
print("="*80)
print("INFERENCE ON FILTERED AUDIO VARIANTS")
print("="*80)

all_results = {}
for filter_name in ['original', 'combined', 'pre_emphasis', 'spectral_subtraction', 'wiener_filter']:
    results = inference_on_filtered_dataset(filter_name, best_model, max_samples=50)
    all_results[filter_name] = results

print("\n" + "="*80)
print("INFERENCE SUMMARY")
print("="*80)

for filter_name, results in all_results.items():
    if results:
        avg_phonemes = np.mean([r['num_phonemes'] for r in results])
        print(f"\n{filter_name:25s}:")
        print(f"  Samples: {len(results)}")
        print(f"  Avg predicted phonemes per utterance: {avg_phonemes:.1f}")
        print(f"  Sample predictions (first 3):")
        for i, res in enumerate(results[:3]):
            print(f"    {res['speaker']}_{res['utterance']}: {' '.join(res['predicted_phonemes'][:10])}...")


INFERENCE ON FILTERED AUDIO VARIANTS
Running inference on original (50 samples)...
  Progress: 5/50
  Progress: 10/50
  Progress: 15/50
  Progress: 20/50
  Progress: 25/50
  Progress: 30/50
  Progress: 35/50
  Progress: 40/50
  Progress: 45/50
  Progress: 50/50
✓ Inference complete on original

Running inference on combined (50 samples)...
  Progress: 5/50
  Progress: 10/50
  Progress: 15/50
  Progress: 20/50
  Progress: 25/50
  Progress: 30/50
  Progress: 35/50
  Progress: 40/50
  Progress: 45/50
  Progress: 50/50
✓ Inference complete on combined

Running inference on pre_emphasis (50 samples)...
  Progress: 5/50
  Progress: 10/50
  Progress: 15/50
  Progress: 20/50
  Progress: 25/50
  Progress: 30/50
  Progress: 35/50
  Progress: 40/50
  Progress: 45/50
  Progress: 50/50
✓ Inference complete on pre_emphasis

Running inference on spectral_subtraction (50 samples)...
  Progress: 5/50
  Progress: 10/50
  Progress: 15/50
  Progress: 20/50
  Progress: 25/50
  Progress: 30/50
  Progress: 3

In [ ]:

def compute_per(reference, hypothesis):
    """
    Compute Phoneme Error Rate (PER) with operation counts.
    
    Args:
        reference: list of phoneme strings (ground truth)
        hypothesis: list of phoneme strings (predicted)
    
    Returns:
        per: error rate (%)
        substitutions: count
        deletions: count
        insertions: count
    """
    n, m = len(reference), len(hypothesis)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    path = [['' for _ in range(m + 1)] for _ in range(n + 1)]
    
    for i in range(n + 1):
        dp[i][0], path[i][0] = i, 'D' * i
    for j in range(m + 1):
        dp[0][j], path[0][j] = j, 'I' * j
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if reference[i-1] == hypothesis[j-1]:
                dp[i][j] = dp[i-1][j-1]
                path[i][j] = path[i-1][j-1]
            else:
                if dp[i-1][j-1] <= dp[i-1][j] and dp[i-1][j-1] <= dp[i][j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                    path[i][j] = path[i-1][j-1] + 'S'
                elif dp[i-1][j] <= dp[i][j-1]:
                    dp[i][j] = dp[i-1][j] + 1
                    path[i][j] = path[i-1][j] + 'D'
                else:
                    dp[i][j] = dp[i][j-1] + 1
                    path[i][j] = path[i][j-1] + 'I'
    
    ops = path[n][m]
    s, d, i = ops.count('S'), ops.count('D'), ops.count('I')
    per = (s + d + i) / len(reference) * 100 if reference else 0.0
    
    return per, s, d, i


ref = ['aa', 'b', 'ah', 'sh']
hyp = ['aa', 'b', 'er', 'sh', 'eh']
per, sub, dele, inse = compute_per(ref, hyp)
print(f"Example: PER={per:.1f}%, S={sub}, D={dele}, I={inse}")



Example: PER=50.0%, S=1, D=0, I=1


In [ ]:

print("\n" + "="*80)
print("PER COMPARISON: FILTER PREDICTIONS vs ORIGINAL PREDICTIONS")
print("="*80)
print("(No ground truth data used - direct prediction comparison)\n")

original_predictions = [r['predicted_phonemes'] for r in all_results['original']]

results_table = []

for filter_name in ['combined', 'pre_emphasis', 'spectral_subtraction', 'wiener_filter']:
    filter_predictions = [r['predicted_phonemes'] for r in all_results[filter_name]]
    
    per_scores = []
    total_s, total_d, total_i = 0, 0, 0
    
    for orig_pred, filt_pred in zip(original_predictions, filter_predictions):
        per, s, d, i = compute_per(orig_pred, filt_pred)
        per_scores.append(per)
        total_s += s
        total_d += d
        total_i += i
    
    mean_per = np.mean(per_scores)
    std_per = np.std(per_scores)
    
    results_table.append({
        'filter': filter_name,
        'mean_per': mean_per,
        'std_per': std_per,
        'total_s': total_s,
        'total_d': total_d,
        'total_i': total_i,
    })

print(f"{'Filter':<25s} {'Mean PER':>12s} {'Std':>8s} {'S':>6s} {'D':>6s} {'I':>6s}")
print("-" * 75)

for row in results_table:
    print(f"{row['filter']:<25s} {row['mean_per']:>11.2f}% {row['std_per']:>7.2f}% {row['total_s']:>6d} {row['total_d']:>6d} {row['total_i']:>6d}")

print("\n" + "="*80)
print("RANKING (Lower PER = More Similar to Original)")
print("="*80)

sorted_results = sorted(results_table, key=lambda x: x['mean_per'])
for rank, row in enumerate(sorted_results, 1):
    print(f"{rank}. {row['filter']:<25s} PER={row['mean_per']:>6.2f}%  (S={row['total_s']}, D={row['total_d']}, I={row['total_i']})")

print("\n✓ PER comparison complete!")



PER COMPARISON: FILTER PREDICTIONS vs ORIGINAL PREDICTIONS
(No ground truth data used - direct prediction comparison)

Filter                        Mean PER      Std      S      D      I
---------------------------------------------------------------------------
combined                        28.90%    6.02%    595    362    341
pre_emphasis                    21.75%    5.40%    396    329    248
spectral_subtraction            18.77%    5.10%    343    227    269
wiener_filter                    2.09%    2.34%     23     35     34

RANKING (Lower PER = More Similar to Original)
1. wiener_filter             PER=  2.09%  (S=23, D=35, I=34)
2. spectral_subtraction      PER= 18.77%  (S=343, D=227, I=269)
3. pre_emphasis              PER= 21.75%  (S=396, D=329, I=248)
4. combined                  PER= 28.90%  (S=595, D=362, I=341)

✓ PER comparison complete!


The applied filters did not improve the result relative to the original reference. However, Wiener filtering introduced only minimal deviation and therefore was the best-performing filtering method among those tested. Our assumption that the filters did not help because TIMIT is already a clean benchmark dataset. It was created for acoustic-phonetic and ASR research and contains high-quality 16-bit, 16 kHz speech recordings with hand-verified transcriptions. When a dataset is already clean, extra filtering often brings little benefit and may even damage useful phoneme-related information.

---

## STOI (Short-Time Objective Intelligibility)

STOI measures speech intelligibility by comparing original and processed audio in short-time segments. Range: 0–1 (higher = better intelligibility preservation).


In [ ]:
def compute_stoi(reference_audio, filtered_audio, sr):
    """
    Compute STOI (Short-Time Objective Intelligibility).
    
    Args:
        reference_audio: 1D numpy array of original audio
        filtered_audio: 1D numpy array of filtered audio
        sr: sampling rate
    
    Returns:
        stoi_score: float (0–1, higher = better intelligibility)
    """
    if len(reference_audio) != len(filtered_audio):
        min_len = min(len(reference_audio), len(filtered_audio))
        reference_audio = reference_audio[:min_len]
        filtered_audio = filtered_audio[:min_len]
    
    return stoi(reference_audio, filtered_audio, sr)


In [44]:
print("\n" + "="*80)
print("STOI COMPARISON: FILTER AUDIO vs ORIGINAL AUDIO")
print("="*80)
print("(50 samples per filter)\n")

original_dataset = FilteredDataset('original')
num_samples = min(50, len(original_dataset))

stoi_results = []

for filter_name in ['combined', 'pre_emphasis', 'spectral_subtraction', 'wiener_filter']:
    filter_dataset = FilteredDataset(filter_name)
    
    stoi_scores = []
    
    for idx in range(num_samples):
        orig_sample = original_dataset[idx]
        filt_sample = filter_dataset[idx]
        
        orig_audio = orig_sample['audio']
        filt_audio = filt_sample['audio']
        sr = orig_sample['sampling_rate']
        
        score = compute_stoi(orig_audio, filt_audio, sr)
        stoi_scores.append(score)
    
    mean_stoi = np.mean(stoi_scores)
    std_stoi = np.std(stoi_scores)
    median_stoi = np.median(stoi_scores)
    
    stoi_results.append({
        'filter': filter_name,
        'mean': mean_stoi,
        'std': std_stoi,
        'median': median_stoi,
        'min': np.min(stoi_scores),
        'max': np.max(stoi_scores),
    })

print(f"{'Filter':<25s} {'Mean':>10s} {'Std':>8s} {'Median':>10s} {'Min':>8s} {'Max':>8s}")
print("-" * 75)

for row in stoi_results:
    print(f"{row['filter']:<25s} {row['mean']:>10.4f} {row['std']:>8.4f} {row['median']:>10.4f} {row['min']:>8.4f} {row['max']:>8.4f}")

print("\n" + "="*80)
print("RANKING (Higher STOI = Better Intelligibility Preservation)")
print("="*80)

sorted_stoi = sorted(stoi_results, key=lambda x: x['mean'], reverse=True)
for rank, row in enumerate(sorted_stoi, 1):
    print(f"{rank}. {row['filter']:<25s} STOI={row['mean']:.4f} (std={row['std']:.4f})")

print("\n✓ STOI comparison complete!")



STOI COMPARISON: FILTER AUDIO vs ORIGINAL AUDIO
(50 samples per filter)

Filter                          Mean      Std     Median      Min      Max
---------------------------------------------------------------------------
combined                      0.9162   0.0187     0.9158   0.8787   0.9585
pre_emphasis                  0.9959   0.0012     0.9960   0.9918   0.9980
spectral_subtraction          0.9999   0.0001     0.9999   0.9994   1.0000
wiener_filter                 1.0000   0.0000     1.0000   0.9998   1.0000

RANKING (Higher STOI = Better Intelligibility Preservation)
1. wiener_filter             STOI=1.0000 (std=0.0000)
2. spectral_subtraction      STOI=0.9999 (std=0.0001)
3. pre_emphasis              STOI=0.9959 (std=0.0012)
4. combined                  STOI=0.9162 (std=0.0187)

✓ STOI comparison complete!


STOI results show that the original TIMIT speech is already highly intelligible, so filtering does not improve it further. Among the tested methods, Wiener filtering preserved intelligibility best, followed very closely by spectral subtraction and pre-emphasis, while the combined filter caused the largest reduction. Overall, this metric confirms that the filters mostly act as preservation or distortion methods rather than true enhancement on such a clean dataset.